In [ ]:
# ============================
# Google Colab: mBART Fine-tuning for Sinhala Error Correction
# Multi-dataset version (HF datasets list)
# ============================

# ▶️ 0) (Optional) Check GPU
!nvidia-smi -L

In [ ]:
# ▶️ 1) Install dependencies
!pip -q install -U transformers datasets accelerate sentencepiece evaluate sacrebleu huggingface_hub
# Optional (only if you want Weights & Biases logging)
!pip -q install -U wandb

print("✓ Installed")

In [ ]:
# ▶️ 2) Login (Hugging Face + optional W&B)
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami
from huggingface_hub.utils import HfHubHTTPError

load_dotenv()
hf_token = os.environ.get("HF_TOKEN", "").strip()
hf_logged_in = False

try:
    if hf_token:
        # Non-interactive login (best for Colab/CI)
        login(token=hf_token, add_to_git_credential=True)
        user_info = whoami()
        print(f"✓ Hugging Face logged in as: {user_info.get('name', 'unknown')}")
        hf_logged_in = True
    else:
        print("⚠️ HF_TOKEN not found in environment.")
        print("Set it first, e.g. in Colab:")
        print("os.environ['HF_TOKEN'] = 'hf_xxx'")
except HfHubHTTPError as e:
    print("✗ Hugging Face authentication failed.")
    print("Reason:", e)
    print("Please check HF_TOKEN and ensure it has write access to create/push repos.")

use_wandb = False
if use_wandb:
    import wandb
    wandb_key = os.environ.get("WANDB_API_KEY", "")
    if wandb_key:
        wandb.login(key=wandb_key)
        print("✓ W&B logged in via WANDB_API_KEY env")
    else:
        print("⚠️ No WANDB_API_KEY found. Set it or keep use_wandb=False.")

In [ ]:
CONFIG = {
    "model_name": "facebook/mbart-large-50",
    "dataset_ids": [
        "SPEAK-PP/sinhala-spelling-correction-already-corrected-pairs",
        "SPEAK-PP/openslr-sinhala-spelling-correction-prediction-reference",
        "SPEAK-PP/sinhala-itn-dataset",
    ],

    "source_lang": "si_LK",
    "target_lang": "si_LK",

    "max_input_length": 128,
    "max_target_length": 128,

    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "auto_find_batch_size": True,
    "num_epochs": 5,
    "learning_rate": 5e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,

    "push_to_hub": True,
    "hub_model_id": "SPEAK-ASR/mBART-large-50-si-spelling-v4-multi",
}

print("Datasets to load:")
for d in CONFIG["dataset_ids"]:
    print(" -", d)

In [ ]:
# ▶️ 3) Import core libraries
import torch
from datasets import load_dataset, concatenate_datasets, DatasetDict, Features, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ▶️ 4) Load and merge multiple HF datasets
print("[STEP 1] Loading + combining datasets")

TARGET_FEATURES = Features({
    "dyslexic_sentence": Value("large_string"),
    "clean_sentence": Value("large_string"),
})

def get_eval_split(ds_dict):
    if "eval" in ds_dict:
        return ds_dict["eval"]
    if "validation" in ds_dict:
        return ds_dict["validation"]
    return None

def normalize_pair_columns(ds_split):
    """Return split with canonical columns: dyslexic_sentence, clean_sentence."""
    cols = ds_split.column_names

    # input aliases
    src_col = (
        "dyslexic_sentence" if "dyslexic_sentence" in cols else
        "input_text" if "input_text" in cols else
        "textual_format" if "textual_format" in cols else
        None
    )

    # target aliases
    tgt_col = (
        "clean_sentence" if "clean_sentence" in cols else
        "corrected_text" if "corrected_text" in cols else
        "numerical_format" if "numerical_format" in cols else
        None
    )

    if src_col is None or tgt_col is None:
        raise ValueError(f"Could not map columns for split. Found: {cols}")

    if src_col != "dyslexic_sentence":
        ds_split = ds_split.rename_column(src_col, "dyslexic_sentence")
    if tgt_col != "clean_sentence":
        ds_split = ds_split.rename_column(tgt_col, "clean_sentence")

    keep_cols = ["dyslexic_sentence", "clean_sentence"]
    drop_cols = [c for c in ds_split.column_names if c not in keep_cols]
    if drop_cols:
        ds_split = ds_split.remove_columns(drop_cols)

    # Force same schema across all datasets (prevents string vs large_string mismatch)
    ds_split = ds_split.cast(TARGET_FEATURES)

    return ds_split

train_parts = []
eval_parts = []
test_parts = []

for dataset_id in CONFIG["dataset_ids"]:
    ds = load_dataset(dataset_id)
    print(f"\nLoaded {dataset_id} splits: {list(ds.keys())}")

    if "train" in ds:
        train_parts.append(normalize_pair_columns(ds["train"]))
    if "test" in ds:
        test_parts.append(normalize_pair_columns(ds["test"]))

    eval_split = get_eval_split(ds)
    if eval_split is not None:
        eval_parts.append(normalize_pair_columns(eval_split))

if not train_parts or not test_parts:
    raise ValueError("Missing required splits. Need at least train and test across datasets.")

train_merged = concatenate_datasets(train_parts)
test_merged = concatenate_datasets(test_parts)

if eval_parts:
    eval_merged = concatenate_datasets(eval_parts)
else:
    # Fallback if no eval/validation exists in any dataset
    split = train_merged.train_test_split(test_size=0.1, seed=42)
    train_merged = split["train"]
    eval_merged = split["test"]

dataset = DatasetDict({
    "train": train_merged.shuffle(seed=42),
    "eval": eval_merged.shuffle(seed=42),
    "test": test_merged.shuffle(seed=42),
})

print("\n✓ Combined splits:")
for split_name in ["train", "eval", "test"]:
    print(f"  {split_name}: {len(dataset[split_name])}")

print("Columns:", dataset["train"].column_names)
print("Sample:", dataset["train"][0])

In [ ]:
# ▶️ 5) Load tokenizer + model
print("[STEP 2] Loading model/tokenizer:", CONFIG["model_name"])

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=False)
tokenizer.src_lang = CONFIG["source_lang"]
tokenizer.tgt_lang = CONFIG["target_lang"]

model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["model_name"])
lang_token_id = tokenizer.convert_tokens_to_ids([CONFIG["target_lang"]])[0]
model.config.decoder_start_token_id = lang_token_id
model = model.to(device)

print("✓ Model ready")
print("Vocab size:", len(tokenizer))
print("decoder_start_token_id:", model.config.decoder_start_token_id)

In [ ]:
# ▶️ 6) Preprocess + tokenize
print("[STEP 3] Tokenizing & preparing datasets...")

train_cols = dataset["train"].column_names
input_col = "input_text" if "input_text" in train_cols else ("dyslexic_sentence" if "dyslexic_sentence" in train_cols else None)
target_col = "corrected_text" if "corrected_text" in train_cols else ("clean_sentence" if "clean_sentence" in train_cols else None)

if input_col is None or target_col is None:
    raise ValueError(f"Could not find expected columns. Found: {train_cols}")

print("Using columns:")
print("  input_col =", input_col)
print("  target_col =", target_col)

def preprocess_function(examples):
    input_texts, target_texts = [], []

    for i in range(len(examples[input_col])):
        src = examples[input_col][i]
        tgt = examples[target_col][i]

        if src and tgt and str(src).strip() and str(tgt).strip():
            input_texts.append(str(src))
            target_texts.append(str(tgt))

    if len(input_texts) == 0:
        return {"input_ids": [], "attention_mask": [], "labels": []}

    model_inputs = tokenizer(
        input_texts,
        max_length=CONFIG["max_input_length"],
        padding="max_length",
        truncation=True,
    )

    labels = tokenizer(
        text_target=target_texts,
        max_length=CONFIG["max_target_length"],
        padding="max_length",
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset_tok = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=100,
    remove_columns=train_cols,
    desc="Tokenizing",
)

def keep_nonempty(ex):
    return ex["input_ids"] is not None and len(ex["input_ids"]) > 0 and ex["labels"] is not None and len(ex["labels"]) > 0

dataset_tok = dataset_tok.filter(keep_nonempty)

train_dataset = dataset_tok["train"]
eval_dataset = dataset_tok["eval"]
test_dataset = dataset_tok["test"]

print("✓ Split sizes:")
print("  Train :", len(train_dataset))
print("  Eval  :", len(eval_dataset))
print("  Test  :", len(test_dataset))
print("Columns:", train_dataset.column_names)
print("Sample tokenized row:", train_dataset[0])

In [ ]:
# ▶️ 7) Metrics
import numpy as np
import evaluate

sacrebleu = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels_pp = postprocess_text(decoded_preds, decoded_labels)
    bleu = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels_pp)["score"]
    exact = np.mean([p == l[0] for p, l in zip(decoded_preds, decoded_labels_pp)])

    return {"bleu": bleu, "exact_match": float(exact)}

In [ ]:
# ▶️ 8) Training setup
from transformers import set_seed, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, EarlyStoppingCallback

set_seed(42)
output_dir = "/content/mbart-model-v4-multi"

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
report_to = ["wandb"] if use_wandb else []

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=CONFIG["num_epochs"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    push_to_hub=CONFIG["push_to_hub"],
    hub_model_id=CONFIG["hub_model_id"],
    hub_strategy="checkpoint",
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=CONFIG["max_target_length"],
    fp16=use_fp16,
    bf16=use_bf16,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=report_to,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

try:
    trainer = Seq2SeqTrainer(**trainer_kwargs, processing_class=tokenizer)
except TypeError:
    trainer = Seq2SeqTrainer(**trainer_kwargs, tokenizer=tokenizer)

print("✓ Trainer ready")
print("fp16:", use_fp16, "| bf16:", use_bf16)

In [ ]:
# ▶️ 9) Train
print("[STEP 4] Training...")
train_result = trainer.train()
print("✓ Training done")
print("Training loss:", train_result.training_loss)

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to:", output_dir)

In [ ]:
# ▶️ 10) Evaluate on test set
print("[STEP 5] Testing...")
pred_out = trainer.predict(test_dataset)
decoded_preds = tokenizer.batch_decode(pred_out.predictions, skip_special_tokens=True)

labels = np.where(pred_out.label_ids != -100, pred_out.label_ids, tokenizer.pad_token_id)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

print("\nSample predictions:")
for i in range(min(5, len(decoded_preds))):
    print("\n---", i + 1, "---")
    print("PRED:", decoded_preds[i])
    print("GOLD:", decoded_labels[i])
    print("MATCH:", "✓" if decoded_preds[i].strip() == decoded_labels[i].strip() else "✗")

exact = np.mean([p.strip() == l.strip() for p, l in zip(decoded_preds, decoded_labels)])
print("\nExact match (test):", round(float(exact) * 100, 2), "%")

In [ ]:
# ▶️ 11) Push to Hugging Face Hub (if logged in)
from huggingface_hub.utils import HfHubHTTPError

if CONFIG["push_to_hub"] and hf_logged_in:
    print("Pushing to hub:", CONFIG["hub_model_id"])
    try:
        trainer.push_to_hub(
            language="si",
            finetuned_from=CONFIG["model_name"],
            model_name=CONFIG["hub_model_id"],
            dataset=", ".join(CONFIG["dataset_ids"]),
        )
        print("✓ Pushed")
    except HfHubHTTPError as e:
        print("✗ Push failed with Hugging Face API error.")
        print("Reason:", e)
        print("Check that HF_TOKEN is valid and has write permission for this namespace.")
else:
    print("Skipping push_to_hub (not logged in or push_to_hub=False).")